# LLM-as-a-judge (Claude) per generator model, without folds

Each video is classified by Claude in **its own API request** (no memory between classifications): your prompt with the draw's examples inserted at `{examples}` and the video's transcript at `{transcript}`; the answer is constrained to Low / Medium / High. **Zero-shot** (the prompt without examples, or classifications read from `ZERO_SHOT_FILE`) is the point $M = 0$ of the curves.

**Cost**: one request per (draw, video), so the grid of $M$ and the number of draws are smaller than for FGW and embeddings (their first draws: the same examples). Nothing is sent while `RUN_API = False`: the number of requests, their cost and CO$_2$ are estimated first. Every answer is saved (the same file as `human_llm_judge.ipynb`), so a request is never paid twice; within a draw, the prompt up to the transcript is cached by the API.

Everything is computed **separately for each LLM that generated the videos**: the examples and the evaluated videos always come from the same model.

**No cross-validation split** (a fixed split decides which videos serve as examples, and the results depended on it). For each model, label (human or reference) and number of examples $M$:

1. a **draw** takes $M$ random videos per level (Low, Medium, High) among the videos of the model;
2. every video of the model **not in the draw** is classified with these examples;
3. this is repeated `N_DRAWS` times.

For a given video, the examples are a uniform random set of $M$ videos per level among the *other* videos of its model. Scores are pooled over all (draw, video) pairs: the expected score with $M$ random examples; their 95% CI resamples the draws, and the range of a **single** draw shows how much one choice of examples can change the result.

The draws come from `lib/per_model.py` and depend only on the data, `SEED` and $M$: **the FGW, embedding and LLM notebooks use exactly the same examples** (the LLM uses the first `N_DRAWS` of them), and the same scores, plots and tables. The last section compares the methods already run.

**Sensitivity to $M$**: the same for several values of $M$, to see how many examples are needed. **`M_MAIN`** sets the $M$ of the main results: a number (computed even if it is not in `M_VALUES`), or `"auto"`: for each model, the smallest $M$ that is "enough" according to the sensitivity analysis.

In [ ]:
import hashlib
import json
import os
import re
import sys
import time
from concurrent.futures import ThreadPoolExecutor

import anthropic
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from ecologits.tracers.utils import llm_impacts

sys.path.append(os.path.realpath("../lib"))
from evaluation import LEVELS, plot_confusion
from per_model import (compare_methods, draw_example_sets, enough_table, load_data, m_values_to_run, main_m_values,
                       main_results, plot_sensitivity, respect, run_draws, score_draws)
from evaluation import to_level

## Parameters

In [ ]:
LABELS_FILE = "human_labels.csv"
TEXT_COLUMN = "Transcript"
LABEL_COLUMN = "ability_label_name"
REFERENCE_COLUMN = None         # reference (instructed) level; None: a column whose name contains "ref" or "instruct"
GENERATOR_COLUMN = None         # LLM that generated each video; None: a column whose name contains "model", "generator" or "llm"
DIMENSION = "ability"           # names the outputs

M_VALUES = [1, 3, 5, 10]   # numbers of examples per level tested (too large values are skipped per model)
M_MAIN = 10                     # M of the main results: a number, or "auto" (smallest M "enough" per model)
N_DRAWS = 5                    # random draws of the examples, for each model, label and M
SEED = 0                        # same SEED in the three notebooks = same draws
N_BOOTSTRAP = 1000
TOLERANCE = 0.02                # "enough examples": within this of the best score over M
OUTPUT_DIR = "results"

MAX_TARGETS_PER_DRAW = None     # classify at most this many videos per draw (random), to limit the cost; None: all

# Your prompt, in a text file, with {examples} and {transcript} (literal replacement)
PROMPT_FILE = "llm_prompt.txt"
SYSTEM_PROMPT = None
EXAMPLE_TEMPLATE = "Transcript: {text}\nLevel: {level}"
EXAMPLE_SEPARATOR = "\n\n"
NO_EXAMPLES_TEXT = ""           # what replaces {examples} in the zero-shot prompt

# Zero-shot classifications already made (csv with the transcript and the level); None: requested from the API
ZERO_SHOT_FILE = None
ZERO_SHOT_TEXT_COLUMN = None    # None: TEXT_COLUMN
ZERO_SHOT_LEVEL_COLUMN = None   # None: "LLMrating", else a column whose name contains "pred", "rating" or "level"

MODEL = "claude-opus-5-5"
EFFORT = "medium"               # low | medium | high | xhigh | max
MAX_TOKENS = 8000               # includes the model's thinking
USE_FALLBACK = True             # if Claude declines a request, the API re-runs it on Anthropic's recommended fallback model
MAX_WORKERS = 8
RUN_API = False                 # set to True to send the requests (check the estimates first)

PRICES = {"input": 4.0, "cache_write": 5.0, "cache_read": 0.2, "output": 20.0}   # $ per million tokens (claude-opus-5-5)
OUTPUT_TOKENS_GUESS = 400       # per request, thinking included (rough)
ELECTRICITY_MIX_ZONE = "USA"    # EcoLogits electricity mix (e.g. "USA", "EEE" for Europe, "WOR")
LATENCY_GUESS = 10.0            # seconds per request, for the estimate before sending

## Labeled videos and generator models

In [ ]:
df, LABELS, GENERATORS, counts, max_m = load_data(LABELS_FILE, TEXT_COLUMN, LABEL_COLUMN, REFERENCE_COLUMN, GENERATOR_COLUMN)
M_RUN = m_values_to_run(M_VALUES, M_MAIN)
print("largest usable M per model and label:", max_m)
os.makedirs(OUTPUT_DIR, exist_ok=True)
counts

## Zero-shot classifications from a file

Only with `ZERO_SHOT_FILE`: rows matched to the labeled videos by their text (spaces normalized).

In [ ]:
zero_shot_from_file = None
if ZERO_SHOT_FILE:
    zero = pd.read_csv(ZERO_SHOT_FILE)
    text_column = ZERO_SHOT_TEXT_COLUMN or TEXT_COLUMN
    level_column = ZERO_SHOT_LEVEL_COLUMN or ("LLMrating" if "LLMrating" in zero.columns else next(
        c for c in zero.columns if c != text_column and re.search("pred|rating|level", c, re.IGNORECASE)))
    normalize = lambda text: " ".join(str(text).split())
    levels = dict(zip(zero[text_column].map(normalize), zero[level_column].map(to_level)))
    zero_shot_from_file = pd.DataFrame({"id": df["id"], "pred": df["text"].map(normalize).map(levels)})
    print(f"{zero_shot_from_file['pred'].notna().sum()} of {len(df)} videos found with a level in {ZERO_SHOT_FILE}")

## Prompt

In [ ]:
with open(PROMPT_FILE, encoding="utf-8") as f:
    TEMPLATE = f.read()
assert "{transcript}" in TEMPLATE, "the prompt must contain {transcript}"
texts = dict(zip(df["id"], df["text"]))


def format_examples(chosen, seed):
    # The draw's examples ({level: ids}) written with EXAMPLE_TEMPLATE, in a seeded random order.
    if chosen is None:
        return NO_EXAMPLES_TEXT
    pairs = [(i, level) for level in LEVELS for i in chosen[level]]
    order = np.random.default_rng(seed).permutation(len(pairs))
    return EXAMPLE_SEPARATOR.join(EXAMPLE_TEMPLATE.replace("{text}", texts[pairs[k][0]]).replace("{level}", pairs[k][1])
                                  for k in order)


def build_prompt(examples_text, transcript):
    # The prompt split in two: the part before the transcript (cached) and the rest.
    before, after = TEMPLATE.replace("{examples}", examples_text).split("{transcript}", 1)
    return before, transcript + after

## Requests

In [ ]:
SCHEMA = {"type": "object", "properties": {"level": {"type": "string", "enum": LEVELS}},
          "required": ["level"], "additionalProperties": False}
ANSWERS_FILE = os.path.join(OUTPUT_DIR, f"llm_answers_{DIMENSION}.jsonl")  # shared with human_llm_judge.ipynb


def request_key(before, after):
    return hashlib.sha1(json.dumps([MODEL, EFFORT, SYSTEM_PROMPT, before, after]).encode()).hexdigest()


answers = {}
if os.path.exists(ANSWERS_FILE):
    with open(ANSWERS_FILE, encoding="utf-8") as f:
        for line in f:
            record = json.loads(line)
            answers[record["key"]] = record
print(f"{len(answers)} saved answers")


def ask(before, after):
    # One independent request: the level chosen by Claude, or None (refusal or error).
    content = [{"type": "text", "text": before, "cache_control": {"type": "ephemeral"}}] if before else []
    content.append({"type": "text", "text": after})
    kwargs = dict(model=MODEL, max_tokens=MAX_TOKENS, messages=[{"role": "user", "content": content}],
                  output_config={"effort": EFFORT, "format": {"type": "json_schema", "schema": SCHEMA}})
    if SYSTEM_PROMPT:
        kwargs["system"] = SYSTEM_PROMPT
    if USE_FALLBACK:
        kwargs.update(betas=["server-side-fallback-2026-07-01"], fallbacks="default")
    start = time.perf_counter()
    try:
        response = client.beta.messages.create(**kwargs)
    except anthropic.APIStatusError as error:  # not saved, so that it is retried at the next run
        return {"level": None, "stop_reason": f"error {error.status_code}: {error.message}", "save": False}
    level = None
    if response.stop_reason not in ("refusal", "max_tokens"):
        text = next((block.text for block in response.content if block.type == "text"), None)
        level = json.loads(text).get("level") if text else None
    usage = response.usage
    return {"level": level, "stop_reason": response.stop_reason, "served_by": response.model,
            "input_tokens": usage.input_tokens, "output_tokens": usage.output_tokens,
            "cache_read": usage.cache_read_input_tokens or 0, "cache_write": usage.cache_creation_input_tokens or 0,
            "latency": time.perf_counter() - start, "save": True}


def run_requests(jobs):
    # Sends the requests not answered yet; within a group sharing the same prompt start, the first one
    # alone (it writes the cache), then the others in parallel (they read it).
    todo = {}
    for job in jobs:
        if job["key"] not in answers:
            todo.setdefault(job["before"], {})[job["key"]] = job
    total, done = sum(len(g) for g in todo.values()), 0
    with open(ANSWERS_FILE, "a", encoding="utf-8") as out, ThreadPoolExecutor(MAX_WORKERS) as pool:
        for before, group in todo.items():
            group = list(group.values())
            results = [ask(before, group[0]["after"])]
            results += list(pool.map(lambda job: ask(before, job["after"]), group[1:]))
            for job, result in zip(group, results):
                if result.pop("save"):
                    record = {"key": job["key"], **result}
                    answers[job["key"]] = record
                    out.write(json.dumps(record) + "\n")
                else:
                    print(job["id"], result["stop_reason"])
            out.flush()
            done += len(group)
            print(f"  {done} / {total} requests")

## Jobs, cost and CO$_2$ estimates

The draws of `draw_example_sets`: the first `N_DRAWS` draws of the FGW and embedding notebooks (same `SEED`).

In [ ]:
jobs = []
for generator in GENERATORS:
    part = df[df["generator"] == generator]
    for label in LABELS:
        for m in M_RUN:
            if m > max_m[(generator, label)]:
                continue
            for draw, chosen in enumerate(draw_example_sets(df, LABELS, GENERATORS, generator, label, m, N_DRAWS, SEED)):
                used = set(np.concatenate(list(chosen.values())))
                targets = [i for i in part["id"] if i not in used]
                if MAX_TARGETS_PER_DRAW is not None and len(targets) > MAX_TARGETS_PER_DRAW:
                    rng = np.random.default_rng([SEED, m, GENERATORS.index(generator), list(LABELS).index(label), draw])
                    targets = list(rng.choice(targets, MAX_TARGETS_PER_DRAW, replace=False))
                examples_text = format_examples(chosen, [SEED, m, draw])
                for i in targets:
                    before, after = build_prompt(examples_text, texts[i])
                    jobs.append({"generator": generator, "examples": label, "M": m, "draw": draw, "id": i,
                                 "before": before, "after": after, "key": request_key(before, after)})
if zero_shot_from_file is None:
    for i, generator in zip(df["id"], df["generator"]):
        before, after = build_prompt(NO_EXAMPLES_TEXT, texts[i])
        jobs.append({"generator": generator, "examples": None, "M": 0, "draw": 0, "id": i,
                     "before": before, "after": after, "key": request_key(before, after)})

missing = {job["key"]: job for job in jobs if job["key"] not in answers}
groups = {}
for job in missing.values():
    groups.setdefault(job["before"], []).append(job)
tokens = lambda text: len(text) / 4  # rough: about 4 characters per token
cost = sum(tokens(before) * (PRICES["cache_write"] + (len(group) - 1) * PRICES["cache_read"])
           + sum(tokens(job["after"]) for job in group) * PRICES["input"]
           + len(group) * OUTPUT_TOKENS_GUESS * PRICES["output"] for before, group in groups.items()) / 1e6
print(f"{len(jobs)} classifications, {len(missing)} requests not answered yet")
print(f"rough cost of the missing requests with {MODEL}: ${cost:.2f}")


def footprint(requests):
    # Energy (kWh) and emissions (kg CO2e) with EcoLogits, summed as min-max.
    totals, warnings_seen = np.zeros(4), set()
    for model, output_tokens, latency in requests:
        impacts = llm_impacts("anthropic", model, int(output_tokens), float(latency), ELECTRICITY_MIX_ZONE)
        if impacts is None or impacts.energy is None:
            continue
        for k, value in enumerate([impacts.energy.value, impacts.gwp.value]):
            low, high = (value.min, value.max) if hasattr(value, "min") else (value, value)
            totals[2 * k] += low
            totals[2 * k + 1] += high
        warnings_seen |= {w.code for w in (impacts.warnings or [])}
    if warnings_seen:
        print("EcoLogits warnings:", ", ".join(sorted(warnings_seen)))
    return pd.DataFrame({"min": totals[0::2], "max": totals[1::2]}, index=["energy (kWh)", "emissions (kg CO2e)"])


print(f"rough energy and emissions ({ELECTRICITY_MIX_ZONE}, {OUTPUT_TOKENS_GUESS} output tokens and "
      f"{LATENCY_GUESS:.0f} s per request):")
footprint([(MODEL, OUTPUT_TOKENS_GUESS, LATENCY_GUESS)] * len(missing))

In [ ]:
if missing and not RUN_API:
    raise RuntimeError("Check the estimates above, then set RUN_API = True and rerun to send the requests.")
if missing:
    client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY (or an `ant auth login` profile)
    run_requests(missing.values())

## Predictions

Zero-shot is $M = 0$ (a single draw: its CI resamples the videos), shown in both rows of the plots.

In [ ]:
rows = [{"draw": job["draw"], "id": job["id"], "generator": job["generator"], "examples": job["examples"], "M": job["M"],
         "pred": answers.get(job["key"], {}).get("level"), "served_by": answers.get(job["key"], {}).get("served_by")}
        for job in jobs]
if zero_shot_from_file is not None:
    rows += [{"draw": 0, "id": i, "generator": g, "examples": None, "M": 0, "pred": p, "served_by": "from file"}
             for i, g, p in zip(df["id"], df["generator"], zero_shot_from_file["pred"])]
draws = pd.DataFrame(rows)
zero_shot = draws[draws["M"] == 0]
draws = pd.concat([draws[draws["M"] > 0], *[zero_shot.assign(examples=label) for label in LABELS]], ignore_index=True)
draws = draws.merge(df[["id", *LABELS.values()]], on="id")
draws.to_csv(os.path.join(OUTPUT_DIR, f"human_llm_per_model_predictions_{DIMENSION}.csv"), index=False)
print("answers that are not a level:", (~draws["pred"].isin(LEVELS)).sum(), "of", len(draws))
print(draws.drop_duplicates(["id", "M", "draw", "examples"])["served_by"].value_counts(dropna=False).to_string())

## Scores with $M$ random examples

For each model, examples label, $M$ and label evaluated against: accuracy and quadratic-weighted $\kappa$ over all (draw, video) pairs, their 95% CI over draws, and the 5–95% range of the accuracy of a single draw. The `all` row pools the models (draw $d$ of each model together).

In [ ]:
scores = score_draws(draws, LABELS, GENERATORS, N_BOOTSTRAP, SEED)
scores.to_csv(os.path.join(OUTPUT_DIR, f"human_llm_per_model_scores_{DIMENSION}.csv"), index=False)
scores.round(3)

## Sensitivity to the number of examples

Accuracy and $\kappa$ as a function of $M$ (examples per level), one line per model; the band is the 95% CI over draws. Rows: label of the examples; columns: label evaluated against.

In [ ]:
plot_sensitivity(scores, LABELS, GENERATORS, "accuracy", "accuracy", 1 / len(LEVELS),
                 os.path.join(OUTPUT_DIR, f"human_llm_per_model_sensitivity_accuracy_{DIMENSION}.png"), "LLM-as-a-judge")
plot_sensitivity(scores, LABELS, GENERATORS, "kappa", "quadratic-weighted kappa", 0,
                 os.path.join(OUTPUT_DIR, f"human_llm_per_model_sensitivity_kappa_{DIMENSION}.png"), "LLM-as-a-judge")

### How many examples are enough?

For each model, examples label and label evaluated against: the smallest $M$ whose score is within `TOLERANCE` of the best score over the tested $M$, and the spread of a single draw at that $M$. If the best score is reached at the largest $M$ available (`still rising`), more examples might still help.

In [ ]:
enough = enough_table(scores, TOLERANCE)
enough.to_csv(os.path.join(OUTPUT_DIR, f"human_llm_per_model_enough_{DIMENSION}.csv"), index=False)
enough.round(3)

## Main results with `M_MAIN` examples

For each model, the scores with `M_MAIN` examples per level (the largest $M$ computed that does not exceed it; with `"auto"`, the $M$ "enough" for that model), and the confusion matrices over all (draw, video) pairs.

In [ ]:
main_keys = main_m_values(M_MAIN, draws, enough, GENERATORS, LABELS)
print("M used:", {f"{g}, {label} examples": m for (g, label), m in main_keys.items()})
main, main_scores = main_results(draws, scores, main_keys)
main_scores.to_csv(os.path.join(OUTPUT_DIR, f"human_llm_per_model_main_{DIMENSION}.csv"), index=False)
main_scores.pivot_table(index=["examples", "evaluated against"], columns="generator",
                        values=["M", "accuracy [95% CI]", "kappa [95% CI]"], aggfunc="first")

In [ ]:
readable = main[main["pred"].isin(LEVELS)]
plot_confusion({(f"vs {target}", f"examples: {source}"): part.assign(true=part[column])
                for target, column in LABELS.items() for source, part in readable.groupby("examples")},
               "true", os.path.join(OUTPUT_DIR, f"human_llm_per_model_confusion_{DIMENSION}.png"))
plt.show()

## Does each LLM respect the requested level?

For each model: the share of its videos whose level found by the method equals the reference level, with examples labeled by humans or by the reference (main $M$, over all draws; 95% CI over draws), and the share whose **human** level equals the reference level (95% CI over videos). $\chi^2$: equal shares across models, with one prediction per video (the level nearest on average over the draws, or the most frequent one).

In [ ]:
respect_table, respect_tests = respect(df, LABELS, GENERATORS, main, main_scores, N_BOOTSTRAP, SEED, "LLM-as-a-judge", DIMENSION,
                                       os.path.join(OUTPUT_DIR, f"human_llm_per_model_respect_{DIMENSION}.png"), extra_modes={"zero-shot": zero_shot.merge(df[["id", "reference"]], on="id")} if "reference" in LABELS else None)
if respect_table is not None:
    respect_table.to_csv(os.path.join(OUTPUT_DIR, f"human_llm_per_model_respect_{DIMENSION}.csv"), index=False)

## Comparison of the methods

The sensitivity curves (all models pooled) and main scores of the per-model notebooks already run with the same parameters (`human_fgw_per_model`, `human_embeddings_per_model`, `human_llm_judge_per_model`).

In [ ]:
compare_methods(OUTPUT_DIR, DIMENSION, LABELS, "accuracy", os.path.join(OUTPUT_DIR, f"per_model_comparison_{DIMENSION}.png"))

## CO$_2$ of the API requests

EcoLogits estimate for the requests of this notebook that were answered (min–max, usage and manufacturing of the hardware), from the model that answered, its output tokens and the measured duration, with the electricity mix `ELECTRICITY_MIX_ZONE`. Not counted: the zero-shot classifications read from a file.

In [ ]:
used = [answers[job["key"]] for job in {job["key"]: job for job in jobs}.values()
        if job["key"] in answers and "latency" in answers[job["key"]]]
if used:
    co2 = footprint([(a["served_by"], a["output_tokens"], a["latency"]) for a in used])
    co2["per request (min)"], co2["per request (max)"] = co2["min"] / len(used), co2["max"] / len(used)
    co2.to_csv(os.path.join(OUTPUT_DIR, f"human_llm_per_model_co2_{DIMENSION}.csv"))
    print(f"{len(used)} requests, {ELECTRICITY_MIX_ZONE} electricity")
    display(co2)